# Importing Libraries

In [1]:
import pandas as pd
from openai import OpenAI
import json
import os

# Importing Data

In [18]:
file_path = r"sales_database.xlsx"

In [23]:
orders=pd.read_excel(file_path, sheet_name="Orders")
customers=pd.read_excel(file_path, sheet_name="Customers")
products=pd.read_excel(file_path, sheet_name="Products")
payments=pd.read_excel(file_path, sheet_name="Payments")

In [27]:
orders.head(2)

,order_id,customer_id,product_id,quantity,order_date,status
0,O101,C101,P101,10,2026-10-01,Delivered
1,O102,C102,P102,5,2026-10-03,Pending


In [28]:
customers.head(2)

,customer_id,customer_name,city,segment
0,C101,Acme Healthcare,Bengaluru,Enterprise
1,C102,Nova Systems,Hyderabad,Enterprise


In [29]:
products.head(2)

,product_id,product_name,unit_price
0,P101,Analytics Platform,5000
1,P102,AI Assistant License,12000


In [30]:
payments.head(2)

,payment_id,order_id,amount_paid,due_date,payment_status
0,PAY101,O101,50000,2026-10-10,Paid
1,PAY102,O102,20000,2026-10-15,Partial


In [65]:
orders[orders['customer_id']=='C101']

,order_id,customer_id,product_id,quantity,order_date,status
0,O101,C101,P101,10,2026-10-01,Delivered
2,O103,C101,P103,3,2026-10-05,Delivered


# Create Tools

In [139]:
def get_customer_details(customer_id):
    result = customers[customers['customer_id'] == customer_id]
    if result.empty:
        return {"error":'Customer not found'}
    
    return result.to_dict(orient = 'records')

def get_order_details(order_id):
    result = orders[orders['order_id'] == order_id]

    if result.empty:
        return {"error":"Order ID not found"}
    return result.to_dict(orient='records')

def get_product_details(product_id):
    result = products[products['product_id'] == product_id]

    if result.empty:
            return {"error":"Product ID not found"}
    
    return result.to_dict(orient='records')

def get_customer_orders(customer_id):
    result = orders[orders['customer_id'] == customer_id]

    if result.empty:
            return {"error":"Customer ID not found"}
    
    return result.to_dict(orient='records')


def get_payment_status(order_id):
    result = payments[payments['order_id'] == order_id]

    if result.empty:
            return {"error":"Order ID not found"}
        
    return result.to_dict(orient = 'records')


def calculate_order_value(quantity, unit_price):
    total = quantity*unit_price
    return {'quantity': quantity,
            'unit_price': unit_price,
            'total':total}

In [140]:
get_payment_status(order_id='O101010')

{'error': 'Order ID not found'}

In [141]:
calculate_order_value(15, 25)

{'quantity': 15, 'unit_price': 25, 'total': 375}

In [172]:
get_customer_details('C101')

[{'customer_id': 'C101',
  'customer_name': 'Acme Healthcare',
  'city': 'Bengaluru',
  'segment': 'Enterprise'}]

In [143]:
get_order_details(order_id='O1010')

{'error': 'Order ID not found'}

In [146]:
get_product_details(product_id="P1010")

{'error': 'Product ID not found'}

In [111]:
# print(get_customer_details('C101').to_string())

In [112]:
# print(get_customer_details('C101').to_csv())

In [113]:
# print(get_customer_details('C101').to_markdown())

In [120]:
# print(get_customer_details('C101').to_html())

### Create Tool Schema

In [147]:
def create_tool(name, description, properties, required):
    return {
        "type":"function",
        "function":{
            "name": name,
            "description": description,
            "parameters":{
                "type":"object",
                "properties":properties,
                "required":required,
                "additionalProperties":False
            }
        }
    }

In [148]:
tools=[
        create_tool(
                    name='get_customer_details',
                    description="Get the details of the cutomer based upon Customer ID",
                    properties={'customer_id':{"type":"string"}},
                    required=['customer_id']
                    ),

        create_tool(
                    name='get_customer_orders',
                    description="Get all the orders belong to a customer!",
                    properties={'customer_id':{"type":"string"}},
                    required=['customer_id']
                    ),
            
        create_tool(
                    name='get_order_details',
                    description="Get order information including order quantity using the Order ID",
                    properties={"order_id":{"type":"string"}},
                    required= ['order_id']
                    ),

        create_tool(
                    name="calculate_order_value",
                    description="Calculate the order value from quantity and unit price",
                    properties={
                                "quantity":{"type":"number"},
                                "unit_price":{"type":"number"}
                                },
                    required=['quantity', 'unit_price']
        ),

        create_tool(
                    name="get_payment_status",
                    description="Get the payment status / information from the Order ID",
                    properties={"order_id":{"type":"string"}},
                    required=['order_id']
                    ),

        
        create_tool(
                name="get_product_details",
                description="Get product information and the unit price using Product ID",
                properties={"product_id":{"type":"string"}},
                required=["product_id"]
        )
            
    ]

In [149]:
tools

[{'type': 'function',
  'function': {'name': 'get_customer_details',
   'description': 'Get the details of the cutomer based upon Customer ID',
   'parameters': {'type': 'object',
    'properties': {'customer_id': {'type': 'string'}},
    'required': ['customer_id'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_customer_orders',
   'description': 'Get all the orders belong to a customer!',
   'parameters': {'type': 'object',
    'properties': {'customer_id': {'type': 'string'}},
    'required': ['customer_id'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_order_details',
   'description': 'Get order information including order quantity using the Order ID',
   'parameters': {'type': 'object',
    'properties': {'order_id': {'type': 'string'}},
    'required': ['order_id'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'calculate_order_value',
   'description': 'Calc

In [150]:
len(tools)

6

### Create Function Registry

In [151]:
function_mapping ={
    "get_customer_details":get_customer_details,
    "get_customer_orders":get_customer_orders,
    "get_order_details":get_order_details,
    "get_product_details":get_product_details,
    "get_payment_status":get_payment_status,
    "calculate_order_value":calculate_order_value
}

### Craete a Tool Executor

In [161]:
def execute_tool(tool_call):
    name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print(f"\nTool: {name} is requested with Arguments: {arguments}\n")

    function = function_mapping.get(name)

    if function is None:
        return {"error":f"Unknown Tool {name}"}
    else:
        try:
            result = function(**arguments)
        except Exception as e:
            result = {"error":f"Tool Execution Failed {e}"}

    return {
        "role":"tool",
        "tool_call_id":tool_call.id,
        "content":json.dumps(result, default=str)
    }

# Assistant / Tool Loop

In [162]:
client = OpenAI()

In [165]:
def run_sales_assistant(query, tools, max_rounds=8, model ='gpt-5.4-mini'):

    # Create Message
    messages = [
        {"role":"system",
         "content":"""You are a Sales Operations Assistant.
         Always use the available tools to answer questions about customers, orders, products, order value, payment status.
         Do not invent business data. Treat tool results as the source of truth.
         If any record is missing then clearly mention data not available"""},

         {"role":"user",
          "content":query}
    ]


    for call in range(max_rounds):
        print(f"API Call: {call+1}")

        response = client.chat.completions.create(
            model = model,
            messages = messages,
            tools = tools
        )

        assistant_message = response.choices[0].message

        # Add Assistant Message to the Messages
        messages.append(assistant_message)

        # Check for tool calls
        if not assistant_message.tool_calls:
            return assistant_message.content

        for tool_call in assistant_message.tool_calls:
            tool_result_message = execute_tool(tool_call=tool_call)
            messages.append(tool_result_message)


    print('Maximum rounds are exhausted')

In [170]:
query = """Share the order details of customer C101. I want to know what product he orderd.
Also want to know the quantity and if it has been delivered or not. Also mention the payment status.
Kindly arrange the output in tabular format so that it is easy to read and understand."""

In [171]:
result=run_sales_assistant(query=query,
                           tools=tools)
print(result)

API Call: 1

Tool: get_customer_orders is requested with Arguments: {'customer_id': 'C101'}

API Call: 2

Tool: get_product_details is requested with Arguments: {'product_id': 'P101'}


Tool: get_payment_status is requested with Arguments: {'order_id': 'O101'}


Tool: get_product_details is requested with Arguments: {'product_id': 'P103'}


Tool: get_payment_status is requested with Arguments: {'order_id': 'O103'}

API Call: 3
Here are the order details for customer **C101**:

| Order ID | Product ID | Product Name               | Quantity | Delivery Status | Payment Status |
|---------|------------|----------------------------|----------|-----------------|----------------|
| O101    | P101       | Analytics Platform         | 10       | Delivered       | Paid           |
| O103    | P103       | Data Integration Service   | 3        | Delivered       | Paid           |

If you'd like, I can also provide the **order dates** and **payment amounts** in the same table.


In [153]:
name = "some error"
d={"error":f"Unknown Tool {name}"}
print(d)

{'error': 'Unknown Tool some error'}


In [155]:
json.dumps(d, default=str)

'{"error": "Unknown Tool some error"}'

In [156]:
str(d)

"{'error': 'Unknown Tool some error'}"